In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore, ScaleInvariantSignalDistortionRatio


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_3_heads, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver2_abs_pha, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_3_heads, use_pcs, inv_pcs, model_eval_old

import matplotlib.pyplot as plt
import random

In [2]:
TEST_DIR = os.path.join("data", "DS_10283_2791", "clean_testset_wav")
TEST_NOISE_DIR = os.path.join("data", "DS_10283_2791", "noisy_testset_wav")
NOISE_DIR = os.path.join("data", "demand_test")

CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import random

SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

In [4]:
from src.fspen_configs import *

configs = TrainConfig_48kHz_enc_ext()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension_ver2_abs_pha_TRA(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48kHz_enc_ext_tra#0.pt"), map_location="cpu",  weights_only=False)

64


In [5]:
fspen.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [6]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate
BATCH_SIZE = 8 # 32

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cuda time!!!


In [7]:
rir_dict = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
dataset = TRUNetDataset(TEST_DIR, sr=SR, noise_dir=NOISE_DIR, rir_dir=rir_dict, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False)
dataset.set_epoch(99)

180
12


In [8]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [9]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [10]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    input_signal, target_signal, noise, rir = zip(*batch)
        
    max_len_s = max(s.shape[-1] for s in input_signal)
    
    padded_input = torch.zeros(len(input_signal), max_len_s)
    padded_target = torch.zeros(len(target_signal), max_len_s)
    
    for i, s in enumerate(input_signal):
        padded_input[i, :s.shape[-1]] = s
        padded_target[i, :s.shape[-1]] = target_signal[i]

    return padded_input, padded_target


def collate_fn(batch):
    
    padded_input, padded_target = pad_sequence(batch)
        
    padded_input = padded_input.reshape(-1, padded_input.shape[-1])
    padded_target = padded_target.reshape(-1, padded_input.shape[-1])

    return padded_input, padded_target

In [11]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [12]:
import time

def check_stream_inference(model, loader, window_size = 1 * SR // 4, device="cpu"):
    model.eval()

    result_nisqa_full = []
    result_rtf_full = []
    result_nisqa_chunk = []
    result_rtf_chunk = []
    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)
    
            start_time = time.time()
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            ) 

            # spec = use_pcs(spec, N_FFTS)
            
            output, _ = model_eval(model, spec, configs, device, hid_size=HID_SIZE)

            # output = inv_pcs(output.abs(), output.angle())

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            end_time = time.time()
            
            result_rtf_full.append((signal.shape[-1] / SR) / (end_time - start_time))
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            result_nisqa_full.append(nisqa_score)

            h0 = None
            for j in range(0, signal.shape[-1], window_size):
                chunk = signal[..., j:j+window_size]
                
                if chunk.shape[-1] < window_size:
                    continue

                start_time = time.time()
                spec = torch.stft(
                    chunk,
                    n_fft=N_FFTS,
                    hop_length=HOP_LENGTH,
                    # onesided=True,
                    win_length=N_FFTS,
                    window=window,
                    return_complex=True,
                    normalized=True,
                    center=True
                )

                output, h0 = model_eval(model, spec, configs, device, hid_size=HID_SIZE, h0=h0)

                window = vorbis_window(N_FFTS).to(device)
                output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                    window=window,
                                    # onesided=True,
                                    return_complex=False,
                                    normalized=True,
                                    center=True)
                
                end_time = time.time()


                result_rtf_chunk.append((chunk.shape[-1] / SR) / (end_time - start_time))
                # print(output.shape)
                nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

                result_nisqa_chunk.append(nisqa_score)
                

    print(f"Mean nisqa for full audio: ", torch.stack(result_nisqa_full).mean(dim=0))
    print(f"Mean rtf for full audio: ", torch.tensor(result_rtf_full).mean(dim=0), 1 / torch.tensor(result_rtf_full).mean(dim=0))
    print("---" * 10)
    print("Mean nisqa for \"stream\" audio: ", torch.stack(result_nisqa_chunk).mean(dim=0))
    print("Mean rtf for \"stream\" audio: ", torch.tensor(result_rtf_chunk).mean(dim=-1), 1 / torch.tensor(result_rtf_chunk).mean(dim=-1))

    return result_nisqa_full, result_rtf_full, result_nisqa_chunk, result_rtf_chunk

In [13]:
# _ = check_stream_inference(fspen, test_dataloader, device="cpu")

In [14]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to("cuda")
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to("cuda")
sisdr = ScaleInvariantSignalDistortionRatio().to(DEVICE)
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [15]:
from torchaudio.transforms import Resample
from thop import profile


def get_metrics(model, loader, device="cpu"):
    model.eval()
    
    model = model.to(device)
    
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    sisdr_scores = []
    dnsmos_scores = []
    macs_list = []
    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)
    
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            )

            # spec = use_pcs(spec, N_FFTS)
            # print(spec.device)
            output, _ = model_eval(model, spec, configs, device, hid_size=32)

            abs_s = spec.abs()
            input_s_ = torch.permute(torch.view_as_real(spec), dims=(0, 2, 3, 1))
            batch, frames, channels, frequency = input_s_.shape
            abs_s = torch.permute(abs_s, dims=(0, 2, 1))
            abs_s = torch.reshape(abs_s, shape=(batch, frames, 1, frequency))
            h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"]) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

            macs, _ = profile(model.cpu(), inputs=(input_s_.cpu(), abs_s.cpu(), h0), verbose=False)
            macs_list.append(macs / (signal.shape[-1] / SR))

            model = model.to(device)

            # output = inv_pcs(output.abs(), output.angle())

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            output = output / (output.abs().max() / signal.abs().max())
            
            min_l = min(output.shape[-1], signal.shape[-1])
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            
            output = output[..., :min_l]
            target = target[..., :min_l]

            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target = resampler(target.cpu()).cuda()

            stoi_score = stoi(output[..., :min_l], target[..., :min_l])

            sisdr_score = sisdr(output, target)

            srmr_score = srmr(output.detach().cpu())
            dnsmos_score = dnsmos(output.detach())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            sisdr_scores.append(sisdr_score.cpu())
            dnsmos_scores.append(dnsmos_score.cpu())

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dnsmos_scores, "sisdr": sisdr_scores, "macs": macs_list}
        
    return result

In [16]:
metrics = get_metrics(fspen, test_dataloader, device="cuda")

100%|██████████| 824/824 [19:30<00:00,  1.42s/it]


In [17]:
print("NISQA:", torch.vstack(metrics["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics["stoi"]).mean(dim=0))
print("SI-SDR:", -torch.vstack(metrics["sisdr"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics["dnsmos"]).mean(dim=0))
print("MACs:", sum(metrics["macs"]) / len(metrics["macs"]))

NISQA: tensor([3.737, 3.919, 3.733, 3.866, 3.945])
PESQ: tensor([2.428])
SRMR: tensor([7.875])
STOI: tensor([0.809])
SI-SDR: tensor([11.415])
DNSMOS: tensor([3.304, 3.177, 3.810, 2.833], dtype=torch.float64)
MACs: 1165752782.1968968


In [18]:
fspen

FullSubPathExtension_ver2_abs_pha_TRA(
  (full_band_encoder): TRAFullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
    (tra_conv1): TRAEncoderBlock(
      (conv1): Conv1d(16, 48, ke

In [19]:
from thop import profile

window = vorbis_window(N_FFTS)

input_spec = torch.stft(
            torch.ones(1, 48_000),
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

input_spec_ = input_spec.to("cpu")

# abs_spectrum = input_spec.abs()
# input_spec_ = torch.permute(torch.view_as_real(input_spec), dims=(0, 2, 3, 1))
batch = 1 # , frames, channels, frequency = input_spec_.shape
# abs_spectrum = torch.permute(abs_spectrum, dims=(0, 2, 1))
# abs_spectrum = torch.reshape(abs_spectrum, shape=(batch, frames, 1, frequency))
h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"], device=input_spec.device) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

# # output, hid_out = fspen(input_spec_, abs_spectrum, h0)
# print(input_spec_.shape)

input_spec_ = torch.ones(1, 1, 2, 513)
abs_spectrum = torch.ones(1, 1, 1, 513)

fspen.eval()

start = time.time()
macs, params = profile(fspen.cpu(), inputs=(input_spec_, abs_spectrum, h0))
end = time.time()

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose1d'>.


In [20]:
print("MACs: ", macs)
print("Params: ", params)

MACs:  12404608.0
Params:  274602.0
